In [1]:
import pandas as pd

TWITTER_PATH = (
    "/Users/janneslampe/Desktop/Coding/Master Thesis/Twitter Parliamentarian Database"
)

In [15]:
member_info = TWITTER_PATH + "/2020_member_info.csv"
df = pd.read_csv(member_info, sep=",", encoding="utf-16", dtype={"uid": str}, keep_default_na=False)

/var/folders/yc/yqhl95zn0kg8mmy01bz8fhn00000gn/T/ipykernel_84619/4074623217.py:2: DtypeWarning: Columns (0: is_nationalist) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(member_info, sep=",", encoding="utf-16", dtype={"uid": str}, keep_default_na=False)


In [16]:
df["uid"]

0                  234797704
1                           
2                   15394954
3                   17976923
4                  295685416
                ...         
15352    1109041676042018816
15353              479394745
15354              250740163
15355              435720619
15356               48667895
Name: uid, Length: 15357, dtype: str

In [17]:
import re
import pandas as pd

# 1. Deduplicate column list while preserving order
desired_columns = [
    "name",
    "member_id",
    "party_id",
    "uid",
    "party",
    "name_link",
    "function",
    "region",
    "country",
    "country_id",
    "mp_party_id",
    "id",
    "country_abbr_x",
    "is_current",
    "party_abbr",
    "is_nationalist",
    "political_group",
    "mp_country_x",
    "oecdmember",
    "eumember",
    "edate",
    "date",
    "partyname",
    "partyabbrev",
    "mp_country_party",
    "country_name_short",
    "party_name_short",
    "pg_party_name",
    "party_name_ascii",
    "country_partyabbrev",
]

# Keep only the unique column names
desired_columns = list(dict.fromkeys(desired_columns))

# 2. Handle duplicate column names if pandas added suffixes (e.g., country.1)
# First, remove duplicate columns in the raw merged DataFrame
cleaned_df = df.loc[:, ~df.columns.duplicated()].copy()

# Keep only the requested columns that exist in the DataFrame
available_columns = [
    col for col in desired_columns if col in cleaned_df.columns
]
cleaned_df = cleaned_df[available_columns]


# 3. Clean messy text fields (newlines, excessive whitespace)
def clean_text(val):
    if pd.isna(val):
        return val
    # Replace multiple whitespaces/newlines with a single space
    return re.sub(r"\s+", " ", str(val)).strip()


# Apply cleaning across all string/object columns
object_cols = cleaned_df.select_dtypes(include="object").columns
cleaned_df[object_cols] = cleaned_df[object_cols].map(clean_text)

# 4. Standardize empty strings to NaN
cleaned_df.replace("", pd.NA, inplace=True)

/var/folders/yc/yqhl95zn0kg8mmy01bz8fhn00000gn/T/ipykernel_84619/3239409753.py:61: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  object_cols = cleaned_df.select_dtypes(include="object").columns


,name,member_id,party_id,uid,party,name_link,function,region,country,country_id,...,edate,date,partyname,partyabbrev,mp_country_party,country_name_short,party_name_short,pg_party_name,party_name_ascii,country_partyabbrev
0,austin scott,101,0,234797704,Republican,NaN,Agriculture Armed Services,Georgia,United States,25,...,02/11/1920,192011,Republican Party,Republicans,United States Republican Party,NaN,NaN,NaN,NaN,United States Republicans
1,glenn w. thompson,103,0,NaN,Republican,NaN,Agriculture Education and the Workforce Natura...,Pennsylvania,United States,25,...,02/11/1920,192011,Republican Party,Republicans,United States Republican Party,NaN,NaN,NaN,NaN,United States Republicans
2,robert e. latta,107,0,15394954,Republican,NaN,Energy and Commerce,Ohio,United States,25,...,02/11/1920,192011,Republican Party,Republicans,United States Republican Party,NaN,NaN,NaN,NaN,United States Republicans
3,cathy mcmorris rodgers,110,0,17976923,Republican,NaN,Energy and Commerce,Washington,United States,25,...,02/11/1920,192011,Republican Party,Republicans,United States Republican Party,NaN,NaN,NaN,NaN,United States Republicans
4,k. michael conaway,114,0,295685416,Republican,NaN,Agriculture Armed Services Intelligence (Perma...,Texas,United States,25,...,02/11/1920,192011,Republican Party,Republicans,United States Republican Party,NaN,NaN,NaN,NaN,United States Republicans
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15352,"Mazón Ramos, José María",16262,602,1109041676042018816,PRC,http://www.congreso.es/portal/page/portal/Cong...,NaN,NaN,Spain,21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15353,"Oramas González-Moro, Ana María",16291,603,479394745,Cca-NC,http://www.congreso.es/portal/page/portal/Cong...,NaN,NaN,Spain,21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15354,"Quevedo Iturbe, Pedro",16314,604,250740163,NC-CCa-PNC,http://www.congreso.es/portal/page/portal/Cong...,NaN,NaN,Spain,21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15355,"Rego Candamil, Néstor",16323,605,435720619,BNG,http://www.congreso.es/portal/page/portal/Cong...,NaN,NaN,Spain,21,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
eu_members = ['Austria', 'Belgium', 'Denmark', 'Finland', 'France', 'Germany', 'Greece', 'Ireland', 'Italy', 'Latvia', 'Luxembourg', 'Malta', 'Netherlands', 'Poland', 'Slovenia', 'Spain', 'Sweden', 'European Parliament']
eu_members_df = cleaned_df[cleaned_df["country"].isin(eu_members)]
eu_members_df.to_csv(TWITTER_PATH + "/eu_members.csv", index=False, encoding="utf-8")

In [19]:
tweet_ids = TWITTER_PATH + "/all_tweet_ids.csv"
tweet_ids_df = pd.read_csv(tweet_ids, sep=",", encoding="utf-8")

In [20]:
tweet_ids_df.head(10)

,866224163207483392
0,866224414425219072
1,866224432515252225
2,866224455672049664
3,866224459396599809
4,866224838540763137
5,866224965464600576
6,866225018908405760
7,866225040483913728
8,866225475978354690
9,866225496039796736


In [21]:
tweet_ids_2021 = TWITTER_PATH + "/2021.csv"

column_names = [
    "country",
    "party",
    "name",
    "uid",
    "extra",
    "date",
    "tweet_id",
]

df_2021 = pd.read_csv(
    tweet_ids_2021,
    sep=",",
    encoding="utf-8",
    header=None,
    names=column_names,
    dtype={"tweet_id": str, "uid": str},  # Preserves precision of large IDs
    parse_dates=["date"],  # Parses the date column automatically
)

KeyboardInterrupt: 

In [ ]:
df_2021.head(10)
sorted(df_2021["country"].dropna().unique())
df_2021_eu = df_2021[df_2021["country"].isin(eu_members)]
df_2021_eu

,country,party,name,uid,extra,date,tweet_id
1769023,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2021-01-01 12:23:39,1344982617477820417
1769024,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2021-01-01 12:24:04,1344982724432560128
1769025,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2021-01-01 12:24:51,1344982921153814528
1769026,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2021-01-01 12:26:51,1344983421920096256
1769027,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2021-01-01 12:29:08,1344984000176250880
...,...,...,...,...,...,...,...
7555849,Netherlands,VVD,Dilan Yeşilgöz-Zegerius,99354836,\N,2021-08-10 14:59:44,1425109614463438850
7555850,Netherlands,VVD,Dilan Yeşilgöz-Zegerius,99354836,\N,2021-08-10 18:58:03,1425169592712933376
7555851,Netherlands,VVD,Dilan Yeşilgöz-Zegerius,99354836,\N,2021-08-10 19:31:47,1425178079257899013
7555852,Netherlands,VVD,Dilan Yeşilgöz-Zegerius,99354836,\N,2021-08-10 20:19:03,1425189975591620613


In [ ]:
tweet_ids_2020 = TWITTER_PATH + "/2020.csv"

column_names = [
    "country",
    "party",
    "name",
    "uid",
    "extra",
    "date",
    "tweet_id",
]

df_2020 = pd.read_csv(
    tweet_ids_2020,
    sep=",",
    encoding="utf-8",
    header=None,
    names=column_names,
    dtype={"tweet_id": str, "uid": str},  # Preserves precision of large IDs
    parse_dates=["date"],  # Parses the date column automatically
)

/var/folders/yc/yqhl95zn0kg8mmy01bz8fhn00000gn/T/ipykernel_57755/2273218635.py:13: DtypeWarning: Columns (0: extra) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2020 = pd.read_csv(


In [ ]:
df_2020.head(10)
sorted(df_2020["country"].dropna().unique())
df_2020_eu = df_2020[df_2020["country"].isin(eu_members)]
df_2020_eu

,country,party,name,uid,extra,date,tweet_id
2691540,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2020-01-01 06:41:39,1212262592716128256
2691541,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2020-01-01 06:42:58,1212262923680337920
2691542,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2020-01-01 06:46:24,1212263789246263300
2691543,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2020-01-01 06:46:47,1212263885748756485
2691544,Slovenia,Slovenian Democratic Party Deputy Group,janez janša,258856900,NaN,2020-01-01 06:47:46,1212264132017303554
...,...,...,...,...,...,...,...
11150352,Ireland,Sinn Féin,Aengus Ó Snodaigh,82645739,\N,2020-12-17 11:13:13,1339529075703083009
11150353,Ireland,Sinn Féin,Aengus Ó Snodaigh,82645739,\N,2020-12-18 00:55:51,1339736098159931394
11150354,Ireland,Sinn Féin,Aengus Ó Snodaigh,82645739,\N,2020-12-18 01:00:45,1339737332677828609
11150355,Ireland,Sinn Féin,Aengus Ó Snodaigh,82645739,\N,2020-12-19 11:12:58,1340253789530558466


In [ ]:
# 1. Drop the 'extra' column from both DataFrames
df_2020_eu = df_2020_eu.drop(columns=["extra"], errors="ignore")
df_2021_eu = df_2021_eu.drop(columns=["extra"], errors="ignore")

# 2. Concatenate vertically
df_eu_all = pd.concat([df_2020_eu, df_2021_eu], ignore_index=True)

# 3. Export to a combined CSV
df_eu_all.to_csv(
    TWITTER_PATH + "/eu_tweets_2020_2021.csv",
    index=False,
    encoding="utf-8",
)

# Load previsouly created CSV with all tweet ids and speakers

In [ ]:
import pandas as pd

TWITTER_PATH = (
    "/Users/janneslampe/Desktop/Coding/Master Thesis/Twitter Parliamentarian Database"
)

In [ ]:
eu_members_path = TWITTER_PATH + "/eu_members.csv"
eu_tweets_path = TWITTER_PATH + "/eu_tweets_2020_2021.csv"
parties_path = TWITTER_PATH + "/parties_2021_04_28.csv"
merged_path = TWITTER_PATH + "/merged_tweets_members_parties.csv"

members_df = None # pd.read_csv(eu_members_path, sep=",", dtype={"uid": str, "party_id": str}, encoding="utf-8")
tweets_df = None # pd.read_csv(eu_tweets_path, sep=",", dtype={"uid": str}, encoding="utf-8")

tweets_members_df = pd.read_csv(merged_path, sep=",", dtype={"uid": str, "party_id": str}, encoding="utf-8")
parties_df = pd.read_csv(parties_path, sep=",", dtype={"parlgov_id": str, "mp_party_id": str}, engine="python")

/var/folders/yc/yqhl95zn0kg8mmy01bz8fhn00000gn/T/ipykernel_91844/797607668.py:9: DtypeWarning: Columns (0: region) have mixed types. Specify dtype option on import or set low_memory=False.
  tweets_members_df = pd.read_csv(merged_path, sep=",", dtype={"uid": str, "party_id": str}, encoding="utf-8")


In [ ]:
# 2. Filter required columns from members_df
member_cols = ["name", "party_id", "uid", "party", "region", "country"]
members_sub = members_df[member_cols].copy()
tweets_df["uid"] = (
    tweets_df["uid"].astype(str).str.strip().replace({"nan": None, "": None})
)
members_sub["uid"] = (
    members_sub["uid"].astype(str).str.strip().replace({"nan": None, "": None})
)
tweets_df["name_clean"] = tweets_df["name"].astype(str).str.strip().str.lower()
members_sub["name_clean"] = (
    members_sub["name"].astype(str).str.strip().str.lower()
)
members_by_uid = (
    members_sub.dropna(subset=["uid"])
    .drop_duplicates(subset=["uid"])
    .drop(columns=["name_clean"])
)
members_by_name = (
    members_sub.dropna(subset=["name_clean"])
    .drop_duplicates(subset=["name_clean"])
    .drop(columns=["uid"])
)
merged_uid = tweets_df.merge(
    members_by_uid, on="uid", how="left", suffixes=("", "_member")
)
unmatched_mask = merged_uid["party_id"].isna()
matched_by_uid = merged_uid[~unmatched_mask]

unmatched_tweets = tweets_df[unmatched_mask]
merged_name = unmatched_tweets.merge(
    members_by_name, on="name_clean", how="left", suffixes=("", "_member")
)
final_df = (
    pd.concat([matched_by_uid, merged_name], ignore_index=True)
    .drop(columns=["name_clean"])
    .reset_index(drop=True)
)
print(
    f"Total tweets: {len(tweets_df)} | Matched: {final_df['party_id'].notna().sum()}"
)
final_df.to_csv(TWITTER_PATH + "/merged_tweets_members_parties.csv", index=False, encoding="utf-8")

Total tweets: 9540369 | Matched: 9540369


In [ ]:
tweets_members_df[tweets_members_df["party_id"] == "215"]


,country,party,name,uid,date,tweet_id,name_member,party_id,party_member,region,country_member
1609512,European Parliament,Green Party,MOLLY SCOTT CATO,726372601,2020-01-01 08:21:48,1212287797232582657,MOLLY SCOTT CATO,215,Green Party,United Kingdom,European Parliament
1609513,European Parliament,Green Party,MOLLY SCOTT CATO,726372601,2020-01-02 08:42:05,1212655287334850560,MOLLY SCOTT CATO,215,Green Party,United Kingdom,European Parliament
1609514,European Parliament,Green Party,MOLLY SCOTT CATO,726372601,2020-01-02 08:43:36,1212655672036339712,MOLLY SCOTT CATO,215,Green Party,United Kingdom,European Parliament
1609515,European Parliament,Green Party,MOLLY SCOTT CATO,726372601,2020-01-02 09:37:16,1212669174436290560,MOLLY SCOTT CATO,215,Green Party,United Kingdom,European Parliament
1609516,European Parliament,Green Party,MOLLY SCOTT CATO,726372601,2020-01-02 11:23:50,1212695996314116096,MOLLY SCOTT CATO,215,Green Party,United Kingdom,European Parliament
...,...,...,...,...,...,...,...,...,...,...,...
7946367,European Parliament,Green Party,Molly SCOTT CATO,726372601,2021-08-21 11:01:31,1429035932661977088,MOLLY SCOTT CATO,215,Green Party,United Kingdom,European Parliament
7946368,European Parliament,Green Party,Molly SCOTT CATO,726372601,2021-08-21 11:10:01,1429038072642027526,MOLLY SCOTT CATO,215,Green Party,United Kingdom,European Parliament
7946369,European Parliament,Green Party,Molly SCOTT CATO,726372601,2021-08-21 12:45:18,1429062052438495234,MOLLY SCOTT CATO,215,Green Party,United Kingdom,European Parliament
7946370,European Parliament,Green Party,Molly SCOTT CATO,726372601,2021-08-21 12:49:48,1429063183575265284,MOLLY SCOTT CATO,215,Green Party,United Kingdom,European Parliament


In [ ]:
# 2. Filter, clean, and rename required columns
parties_sub = parties_df[
    ["party_id", "party", "party_abbr", "country"]
].copy()

parties_sub["party_id"] = (
    parties_sub["party_id"]
    .astype(str)
    .str.strip()
    .replace({"nan": None, "": None})
)

# Deduplicate to prevent row multiplication during merge
parties_sub = parties_sub.dropna(subset=["party_id"]).drop_duplicates(
    subset=["party_id"]
)

# Rename columns to avoid collisions and match your requirements
parties_sub = parties_sub.rename(
    columns={"party": "party_official", "country": "party_country"}
)

In [ ]:
# 3. Merge onto the existing tweets_members_df
final_merged_df = tweets_members_df.merge(
    parties_sub, on="party_id", how="left"
)

print(
    f"Total rows: {len(final_merged_df)} | Matched party details: {final_merged_df['party_abbr'].notna().sum()}"
)
final_merged_df.to_csv(
    TWITTER_PATH + "/final_merged_tweets_members_parties.csv",
    index=False,
    encoding="utf-8",
)

Total rows: 9540369 | Matched party details: 6721449


In [ ]:
import numpy as np

final_merged_df["country"] = np.where(
    (final_merged_df["country"] == "European Parliament")
    & final_merged_df["region"].notna(),
    final_merged_df["region"],
    final_merged_df["country"],
)

In [ ]:
final_merged_df.sample(10, random_state=42)

,country,party,name,uid,date,tweet_id,name_member,party_id,party_member,region,country_member,party_official,party_abbr,party_country
4642477,Poland,Law and Justice,Dobrzyński Leszek,2979663893,2020-01-19 08:50:09,1218817911684567040,leszek dobrzyński,483,Law and Justice,NaN,Poland,Law and Justice,PiS,Poland
7821270,Malta,Partit Laburista,Josianne CUTAJAR,1023896861470547968,2021-06-28 08:34:56,1409430101607993345,Josianne CUTAJAR,233,Partit Laburista,Malta,European Parliament,Partit Laburista,NaN,European Parliament
2343617,Germany,The Left Party,Petra Sitte,293522998,2020-04-11 14:20:10,1248979158669176833,petra sitte,422,The Left Party,NaN,Germany,The Left Party,Linke,Germany
876944,Ireland,Fianna Fáil,thomas byrne,32922034,2020-08-04 20:04:24,1290740389800968192,thomas byrne,126,Fianna Fáil,NaN,Ireland,Fianna Fáil,FF,Ireland
2966084,Italy,SINISTRA ITALIANA - SINISTRA ECOLOGIA LIBERTA ...,nicola fratoianni,425686235,2020-03-04 18:42:04,1235274326137622532,FRATOIANNI Nicola,110,LIBERI E UGUALI,NaN,Italy,LIBERI E UGUALI,NaN,Italy
1686737,Poland,Platforma Obywatelska,JACEK SARYUSZ-WOLSKI,1485429175,2020-01-23 12:31:33,1220323181568892929,JACEK SARYUSZ-WOLSKI,175,Platforma Obywatelska,Poland,European Parliament,Platforma Obywatelska,NaN,European Parliament
7288418,Poland,Law and Justice,iwona michałek,534747356,2021-04-22 21:15:31,1385341519901638664,iwona michałek,483,Law and Justice,NaN,Poland,Law and Justice,PiS,Poland
9493573,Spain,Vox,"Olona Choclán, Macarena",1109573601995440129,2021-01-16 23:46:05,1350590175596990473,"Olona Choclán, Macarena",541,Vox,NaN,Spain,VOX,NaN,European Parliament
5837077,Netherlands,Forum for Democracy,thierry baudet,367703310,2021-07-23 23:20:18,1418712606181838848,thierry baudet,68,Forum for Democracy,NaN,Netherlands,Forum for Democracy,FvD,Netherlands
6844243,Germany,Alliance 90/The Greens,britta haßelmann,1131092102,2021-07-12 16:00:15,1414615597611167744,britta haßelmann,423,Alliance 90/The Greens,NaN,Germany,Alliance 90/The Greens,Grunen,Germany


In [1]:
from datasets.ground_truth.ches_mapping import CHES_COUNTRIES, CHES_PARTIES
import pandas as pd

TWITTER_PATH = (
    "/Users/janneslampe/Desktop/Coding/Master Thesis/Twitter Parliamentarian Database"
)

In [2]:
final_merged_path = TWITTER_PATH + "/final_merged_tweets_members_parties.csv"

final_merged_df = pd.read_csv(
    final_merged_path,
    sep=",",
    dtype={"uid": str, "party_id": str},
    encoding="utf-8",
)

/var/folders/yc/yqhl95zn0kg8mmy01bz8fhn00000gn/T/ipykernel_28220/1268909050.py:3: DtypeWarning: Columns (0: region, 1: party_abbr) have mixed types. Specify dtype option on import or set low_memory=False.
  final_merged_df = pd.read_csv(


In [3]:
import unicodedata
import re

def normalize_text(value):
    if pd.isna(value):
        return ""
    value = unicodedata.normalize("NFKD", str(value))
    value = "".join(char for char in value if not unicodedata.combining(char))
    return re.sub(r"\s+", " ", value).strip().casefold()


def find_ches_party(row):
    country_match = country_lookup.get(row["country"]) or country_lookup.get(row["country_member"])

    if not country_match:
        return pd.Series(
            {
                "ches_country_id": pd.NA,
                "ches_country_abbrev": pd.NA,
                "ches_party_id": pd.NA,
                "ches_party_name_local": pd.NA,
            }
        )

    country_id, country_abbrev = country_match
    party_abbr = normalize_text(row["party_abbr"])
    party_official = normalize_text(row["party_official"])

    candidates = [
        (party_id, details)
        for party_id, details in CHES_PARTIES.items()
        if details["country"] == country_abbrev
    ]

    # Try party abbreviation first.
    for party_id, details in candidates:
        abbreviations = [
            normalize_text(value)
            for value in details["abbrev"].split(";")
        ]
        if party_abbr and any(
            party_abbr == abbreviation or party_abbr in abbreviation
            for abbreviation in abbreviations
        ):
            return pd.Series(
                {
                    "ches_country_id": country_id,
                    "ches_country_abbrev": country_abbrev,
                    "ches_party_id": party_id,
                    "ches_party_name_local": details["name_local"],
                }
            )

    # Then try party abbreviation in the local party name.
    for party_id, details in candidates:
        if party_abbr and party_abbr in normalize_text(details["name_local"]):
            return pd.Series(
                {
                    "ches_country_id": country_id,
                    "ches_country_abbrev": country_abbrev,
                    "ches_party_id": party_id,
                    "ches_party_name_local": details["name_local"],
                }
            )

    # try the official party name in the local party name.
    for party_id, details in candidates:
        if party_official and party_official in normalize_text(details["name_local"]):
            return pd.Series(
                {
                    "ches_country_id": country_id,
                    "ches_country_abbrev": country_abbrev,
                    "ches_party_id": party_id,
                    "ches_party_name_local": details["name_local"],
                }
            )
    # Finally try the official party name in the english party name.
    for party_id, details in candidates:
        if party_official and party_official in normalize_text(details["name_english"]):
            return pd.Series(
                {
                    "ches_country_id": country_id,
                    "ches_country_abbrev": country_abbrev,
                    "ches_party_id": party_id,
                    "ches_party_name_local": details["name_local"],
                }
            )

    return pd.Series(
        {
            "ches_country_id": country_id,
            "ches_country_abbrev": country_abbrev,
            "ches_party_id": pd.NA,
            "ches_party_name_local": pd.NA,
        }
    )


lookup_columns = ["country", "country_member", "party", "party_abbr", "party_official"]

unique_parties = final_merged_df[lookup_columns].drop_duplicates()
country_lookup = {
    details["name"]: (country_id, details["abbrev"])
    for country_id, details in CHES_COUNTRIES.items()
}

party_matches = unique_parties.join(
    unique_parties.apply(find_ches_party, axis=1)
)

ches_integrated_df = final_merged_df.merge(
    party_matches,
    on=lookup_columns,
    how="left",
)

ches_integrated_df[
    [
        "country",
        "party_abbr",
        "party_official",
        "ches_party_id",
        "ches_party_name_local",
    ]
].drop_duplicates()

,country,party_abbr,party_official,ches_party_id,ches_party_name_local
0,Slovenia,SDS,Slovenian Democratic Party Deputy Group,2902,Socialdemokratska stranka Slovenije; Slovenska...
27825,Slovenia,NSI,New Slovenia – Christian Democrats Deputy Group,2905,Nova Slovenija—Krščanska ljudska stranka
31200,Slovenia,SMC,Party of Modern Centre Deputy Group,2911,Stranka Mira Cerarja
66267,Slovenia,DeSUS,Democratic Party of Pensioners of Slovenia Dep...,2906,Demokratična stranka upokojencev Slovenije
66281,Slovenia,SD,Social Democrats Deputy Group,2902,Socialdemokratska stranka Slovenije; Slovenska...
...,...,...,...,...,...
5361560,Spain,Pais,MÁS PAÍS-EQUO,528,Más Pais
5381169,Spain,UPN,UPN,553,Unión del Pueblo Navarro
5930657,Latvia,NaN,Jekabpils,<NA>,NaN
9295854,Belgium,cdH,Centre démocrate humaniste,108,Parti Social Chrétien; Centre Démocrate Humani...


In [4]:
ches_integrated_df.to_csv(
    TWITTER_PATH + "/ches_integrated_tweets_members_parties.csv",
    index=False,
    encoding="utf-8",
)

# Get Twitter Texts

In [9]:
import os
import re
import time
import pandas as pd
import requests
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

TWITTER_PATH = (
    "/Users/janneslampe/Desktop/Coding/Master Thesis/Twitter Parliamentarian Database"
)

In [10]:
ches_integrated_path = TWITTER_PATH + "/ches_integrated_tweets_members_parties.csv"

ches_integrated_df = pd.read_csv(
    ches_integrated_path,
    sep=",",
    dtype={"uid": str, "party_id": str, "tweet_id": str, "ches_party_id": str},
    encoding="utf-8",
)

/var/folders/yc/yqhl95zn0kg8mmy01bz8fhn00000gn/T/ipykernel_59736/1657263494.py:3: DtypeWarning: Columns (0: region, 1: party_abbr) have mixed types. Specify dtype option on import or set low_memory=False.
  ches_integrated_df = pd.read_csv(


In [3]:
def get_tweet_text_oembed(tweet_id):
    url = f"https://publish.twitter.com/oembed?url=https://x.com/i/status/{tweet_id}&omit_script=true"
    response = requests.get(url)

    if response.status_code == 200:
        html = response.json().get("html", "")
        soup = BeautifulSoup(html, "html.parser")
        # Extract main text from the first paragraph tag
        p = soup.find("p")
        return p.get_text() if p else ""
    elif response.status_code == 404:
        return "[Deleted or Private Tweet]"
    return None


In [11]:
# Pre-compile regex to match tweets containing only URLs (single or multiple)
URL_ONLY_PATTERN = re.compile(
    r"^(?:https?://\S+|www\.\S+)(\s+(?:https?://\S+|www\.\S+))*$"
)


def is_valid_tweet_text(text: str | None) -> bool:
    """Check if tweet text is present, not deleted, and not URL-only."""
    if not text or not isinstance(text, str):
        return False

    cleaned = text.strip()
    if not cleaned or cleaned == "[Deleted or Private Tweet]":
        return False

    if URL_ONLY_PATTERN.fullmatch(cleaned):
        return False

    if len(cleaned) < 25:
        return False

    return True


TARGET_COLS = [
    "country",
    "party",
    "name",
    "uid",
    "date",
    "tweet_id",
    "party_id",
    "party_official",
    "ches_party_id",
]
OUTPUT_CSV = "EU_tweets_with_parties_dataset.csv"
OUTPUT_CSV = os.path.join(os.getcwd(), os.path.basename(OUTPUT_CSV))

# 1. Take a sample of 100 rows
sample_df = ches_integrated_df[TARGET_COLS].sample(250).dropna(subset=["ches_party_id"])
sample_df = sample_df[sample_df["ches_party_id"].astype(str).str.strip() != ""]

# 2. Iterate, fetch text, and stream results directly into the CSV
for index, row in tqdm(
    sample_df.iterrows(),
    total=len(sample_df),
    desc="Fetching tweets",
    unit="tweet",
    ):
    try:
        tweet_text = get_tweet_text_oembed(row["tweet_id"])

        if not is_valid_tweet_text(tweet_text):
            continue

        tweet_text = tweet_text.strip()

        # Write valid row to CSV
        row_data = row.to_dict()
        row_data["tweet_text"] = tweet_text

        file_exists = os.path.isfile(OUTPUT_CSV)
        pd.DataFrame([row_data]).to_csv(
            OUTPUT_CSV,
            mode="a",
            index=False,
            header=not file_exists,
        )

    except Exception as e:
        tqdm.write(f"Error f‚etching tweet {row['tweet_id']}: {e}")
        continue

    finally:
        # Guarantee rate-limiting pause occurs on every attempt (including skips/errors)
        time.sleep(0.5)

Fetching tweets: 100%|██████████| 236/236 [04:00<00:00,  1.02s/tweet]
